# 02 Captions and claims: Qwen3-VL-4B, compared with Qwen2.5-VL-3B

Member 4. What this notebook does:
1. Cut each tile into a 3x3 grid and caption every cell (a before crop and an after crop) with **Qwen3-VL-4B-Instruct** on 50 tiles (main run).
2. Caption 20 of the same tiles with **Qwen2.5-VL-3B-Instruct** (comparison, same prompt).
3. Turn every caption line into a `Claim`, check it against the evidence with the deterministic checker, and compare the models.

Needs a **GPU runtime** (Runtime, Change runtime type, T4). Captions are saved to Drive after every tile and a rerun resumes from the saved file; a model is only loaded when tiles are missing. Use Runtime, Restart session, then Run all for a clean GPU.
Before using a model, read its licence on Hugging Face and record it in `data/README.md`. Clear outputs before committing.

In [ ]:
BRANCH = "main"

import os, subprocess, sys

REPO_DIR = "/content/FloodVerify"
if "google.colab" in sys.modules:
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", "https://github.com/aaarrvind/FloodVerify.git", REPO_DIR], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                    "transformers", "accelerate", "pandas", "pyyaml"], check=True)
    REPO = REPO_DIR
else:
    REPO = os.path.abspath("../..")
sys.path.insert(0, os.path.join(REPO, "src"))
print("repo:", REPO)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

ROOT = "/content/drive/MyDrive/MAAI 902 X 901 PROJECT/xbd"
TILES = f"{ROOT}/processed/xbd"              # must hold ALL built tiles (about 5,424); see the check below
EVIDENCE_DIR = f"{ROOT}/processed/evidence_labels"
CAPTIONS = f"{ROOT}/processed/captions"
os.makedirs(CAPTIONS, exist_ok=True)

MAIN_MODEL = "Qwen/Qwen3-VL-4B-Instruct"
COMPARE_MODEL = "Qwen/Qwen2.5-VL-3B-Instruct"
N_PILOT, N_MAIN, N_COMPARE = 10, 50, 20      # pilot tiles tuned the prompt; they are excluded from the main run

In [ ]:
import glob

tile_dirs = [d for d in os.listdir(TILES) if os.path.isdir(f"{TILES}/{d}")] if os.path.isdir(TILES) else []
print("TILES exists:", os.path.isdir(TILES), "| tile folders:", len(tile_dirs))
print("EVIDENCE exists:", os.path.isdir(EVIDENCE_DIR), "| evidence files:", len(glob.glob(f"{EVIDENCE_DIR}/*.json")))

## Model, prompt and captioning functions

The prompt and model ids are part of the experiment: the same prompt goes to both models, and the final settings are written to `captioning.yaml` at the end.

The two models need different input handling. Qwen3-VL accepts the images inside the chat message (`caption_cell`). For Qwen2.5-VL that route silently dropped the images (the model answered that no image was attached), so it uses image placeholders in the template plus `processor(images=...)` (`caption_cell_split`). Both functions stop with an error if no image tokens reach the model.

In [ ]:
import gc, json, torch
from pathlib import Path
from PIL import Image
from transformers import AutoModelForImageTextToText, AutoProcessor

CELL_IDS = [f"r{r}c{c}" for r in range(3) for c in range(3)]

CELL_PROMPT = """Image 1 is BEFORE a flood or storm and Image 2 is AFTER, showing the same small area from above.
Answer with one line only.
If there are no buildings, write: no buildings.
If the area is blank, cloudy or hazy, write: not visible.
If the buildings look the same in both images, write: no damage.
Otherwise write how many of the buildings are damaged and how badly, in the form:
SHARE of the buildings show LEVEL damage.
SHARE is one of: none, few, some, many, most. LEVEL is one of: minor, major, destroyed.
Describe only what can be seen from above."""


def load_captioner(model_id, dtype=torch.float16):
    """Load a VL model and its processor. If outputs are garbled or NaN on a T4, try dtype=torch.bfloat16."""
    try:
        model = AutoModelForImageTextToText.from_pretrained(model_id, dtype=dtype, device_map="auto")
    except TypeError:                       # older transformers call it torch_dtype
        model = AutoModelForImageTextToText.from_pretrained(model_id, torch_dtype=dtype, device_map="auto")
    processor = AutoProcessor.from_pretrained(model_id)
    return model, processor


def free_gpu():
    """Release GPU memory. Delete the model variables first; this only frees what nothing refers to."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def crop_cells(tile_dir, grid=3):
    """Cut the before and after images into a grid; returns {cell_id: (pre_crop, post_crop)}."""
    pre = Image.open(Path(tile_dir) / "pre.png").convert("RGB")
    post = Image.open(Path(tile_dir) / "post.png").convert("RGB")
    w, h = post.size
    out = {}
    for r in range(grid):
        for c in range(grid):
            box = (c * w // grid, r * h // grid, (c + 1) * w // grid, (r + 1) * h // grid)
            out[f"r{r}c{c}"] = (pre.crop(box), post.crop(box))
    return out


def check_images_seen(model, inputs):
    """Fail loudly if the image tokens did not reach the model."""
    token_id = getattr(model.config, "image_token_id", None)
    if token_id is None:
        return
    if int((inputs["input_ids"] == token_id).sum()) == 0:
        raise RuntimeError("No image tokens in the model input: the images were not passed to the model.")


def _generate(model, processor, inputs, max_new_tokens):
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return processor.batch_decode(out[:, inputs["input_ids"].shape[1]:],
                                  skip_special_tokens=True)[0].strip()


def caption_cell(model, processor, pre_img, post_img, prompt=None, max_new_tokens=40):
    """Images go inside the chat message (used for Qwen3-VL)."""
    prompt = prompt or CELL_PROMPT
    messages = [{"role": "user", "content": [
        {"type": "image", "image": pre_img},
        {"type": "image", "image": post_img},
        {"type": "text", "text": prompt}]}]
    inputs = processor.apply_chat_template(messages, add_generation_prompt=True,
                                           tokenize=True, return_dict=True,
                                           return_tensors="pt").to(model.device)
    check_images_seen(model, inputs)
    return _generate(model, processor, inputs, max_new_tokens)


def caption_cell_split(model, processor, pre_img, post_img, prompt=None, max_new_tokens=40):
    """Image placeholders in the template, images passed to the processor (used for Qwen2.5-VL)."""
    prompt = prompt or CELL_PROMPT
    messages = [{"role": "user", "content": [
        {"type": "image"}, {"type": "image"},
        {"type": "text", "text": prompt}]}]
    text = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    inputs = processor(text=[text], images=[pre_img, post_img],
                       return_tensors="pt").to(model.device)
    check_images_seen(model, inputs)
    return _generate(model, processor, inputs, max_new_tokens)


def caption_tiles_by_cell(model, processor, tile_ids, tiles_dir, out_json, cell_fn=caption_cell):
    """Caption every cell of every tile. Resumable: tiles already in out_json are skipped."""
    out = Path(out_json)
    caps = json.loads(out.read_text()) if out.exists() else {}
    for tid in tile_ids:
        if tid in caps:
            continue
        cells = crop_cells(Path(tiles_dir) / tid)
        caps[tid] = {cid: cell_fn(model, processor, *cells[cid]) for cid in CELL_IDS}
        out.write_text(json.dumps(caps, indent=2))
    return {t: caps[t] for t in tile_ids}


def pending(tile_ids, out_json):
    """Tiles that still have no saved captions."""
    p = Path(out_json)
    done = json.loads(p.read_text()) if p.exists() else {}
    return [t for t in tile_ids if t not in done]


def run_captioning(model_id, tile_ids, tiles_dir, out_json, cell_fn):
    """Load the model only if tiles are missing, caption them, and free the GPU afterwards."""
    if pending(tile_ids, out_json):
        model, processor = load_captioner(model_id)
        caption_tiles_by_cell(model, processor, tile_ids, tiles_dir, out_json, cell_fn)
        del model, processor
        free_gpu()
    saved = json.loads(Path(out_json).read_text())
    return {t: saved[t] for t in tile_ids}

## Choose tiles

Only tiles that have an evidence file and built images. The prompt was tuned on a pilot of 10 tiles (kept out of the main run, and read back from `captioning.yaml` when it exists). If the main captions file already exists, its tile list is reused so the selection never changes between runs. The comparison takes 4 tiles per disaster from the main set, so it covers all five disasters.

In [ ]:
import random, collections, yaml

def list_complete_tiles(tiles_dir, evidence_dir):
    ev = {f[:-5] for f in os.listdir(evidence_dir) if f.endswith(".json")}
    return sorted(t for t in os.listdir(tiles_dir)
                  if t in ev and all(os.path.exists(f"{tiles_dir}/{t}/{f}") for f in ("pre.png", "post.png")))


def pick_stratified(tile_ids, n, seed=42):
    """n tiles spread evenly over the disasters."""
    by_d = {}
    for t in tile_ids:
        by_d.setdefault(t.rsplit("_", 2)[0], []).append(t)
    rng = random.Random(seed)
    per = n // len(by_d)
    picked = []
    for d in sorted(by_d):
        picked += rng.sample(by_d[d], min(per, len(by_d[d])))
    return picked


def pick_compare(main_ids, n):
    """The first n // k tiles of each disaster in the main set (k disasters), so every disaster is covered."""
    by_d = {}
    for t in main_ids:
        by_d.setdefault(t.rsplit("_", 2)[0], []).append(t)
    per = n // len(by_d)
    return [t for d in sorted(by_d) for t in by_d[d][:per]]


CFG_PATH = f"{CAPTIONS}/captioning.yaml"
MAIN_FILE = f"{CAPTIONS}/qwen3vl4b_cells_50.json"
CMP_FILE = f"{CAPTIONS}/qwen25vl3b_cells_20_v2.json"

all_tiles = list_complete_tiles(TILES, EVIDENCE_DIR)
print(len(all_tiles), "complete tiles with evidence")
if len(all_tiles) < 5000:
    print("WARNING: far fewer than the expected 5,424 tiles. TILES may point to a partial copy.")

if os.path.exists(CFG_PATH):
    pilot_ids = yaml.safe_load(open(CFG_PATH))["pilot_tiles"]
else:
    pilot_ids = pick_stratified(all_tiles, N_PILOT)

if os.path.exists(MAIN_FILE):
    main_ids = list(json.load(open(MAIN_FILE)).keys())
else:
    pool = [t for t in all_tiles if t not in set(pilot_ids)]
    main_ids = pick_stratified(pool, N_MAIN)
compare_ids = pick_compare(main_ids, N_COMPARE)

print("pilot:  ", len(pilot_ids), "tiles")
print("main:   ", len(main_ids), dict(collections.Counter(t.rsplit("_", 2)[0] for t in main_ids)))
print("compare:", len(compare_ids), dict(collections.Counter(t.rsplit("_", 2)[0] for t in compare_ids)))

## Run 1: Qwen3-VL-4B (main, 50 tiles)

In [ ]:
for _name in ("model", "processor"):        # drop variables left over from earlier cells so the GPU is free
    globals().pop(_name, None)
free_gpu()

caps_main = run_captioning(MAIN_MODEL, main_ids, TILES, MAIN_FILE, caption_cell)
print(len(caps_main), "tiles captioned")
first = main_ids[0]
print(first, caps_main[first])

## Run 2: Qwen2.5-VL-3B (comparison, same prompt, 20 tiles)

Uses `caption_cell_split` and a new output file (`..._v2.json`). The earlier file holds captions from a run in which the images never reached the model, so it must not be reused.

In [ ]:
for _name in ("model", "processor"):
    globals().pop(_name, None)
free_gpu()

caps_cmp = run_captioning(COMPARE_MODEL, compare_ids, TILES, CMP_FILE, caption_cell_split)
print(len(caps_cmp), "tiles captioned")
first = compare_ids[0]
print(first, caps_cmp[first])

### Optional check: both input methods give the same Qwen3-VL captions

Set the flag to True once. It captions one saved tile with `caption_cell_split` and compares it with the saved `caption_cell` captions. If all 9 are identical, the two input methods are equivalent.

In [ ]:
RUN_PATH_CHECK = False

if RUN_PATH_CHECK:
    model, processor = load_captioner(MAIN_MODEL)
    tid = main_ids[0]
    cells = crop_cells(Path(TILES) / tid)
    same = [caption_cell_split(model, processor, *cells[c]) == caps_main[tid][c] for c in CELL_IDS]
    print(sum(same), "of 9 identical")
    del model, processor
    free_gpu()

## Claims and verdicts

Every caption line belongs to one grid cell, so a claim gets its cell id directly. `parse_line` turns a line into a claim list:
`no buildings` and `not visible` make no damage claim, `no damage` becomes (none, damaged), and `SHARE of the buildings show LEVEL damage` becomes (SHARE, LEVEL). `claims_table` runs the checker on them.

In [ ]:
import re, json, collections
import pandas as pd
from floodsitrep.nlp.schema import Claim
from floodsitrep.nlp.checker import verify_claims, class_fraction

PAT = re.compile(r"\b(none|few|some|many|most)\b[^.,;]*?\b(minor|major|destroyed)\b")
VERDICTS = ["supported", "contradicted", "not_verifiable"]
TABLE_COLUMNS = ["model", "tile_id", "cell", "quantity", "damage_class", "verdict",
                 "buildings", "evidence_frac", "text"]


def parse_line(line):
    """One caption line -> (kind, [(quantity, damage_class), ...])."""
    low = line.lower().strip()
    if low.startswith("no buildings"):
        return "no_buildings", []
    if low.startswith("not visible"):
        return "not_visible", []
    if low.startswith("no damage"):
        return "claims", [("none", "damaged")]
    if low.startswith("none") and "damage" in low and not PAT.search(low):
        return "claims", [("none", "damaged")]
    return "claims", [m.groups() for part in low.split(" and ") for m in [PAT.search(part)] if m]


def build_claims(caps):
    """caps = {tile_id: {cell_id: line}} -> [(tile_id, Claim)]."""
    claims = []
    for tid, cells in caps.items():
        for cid, line in cells.items():
            kind, pairs = parse_line(line)
            for q, cls in pairs:
                claims.append((tid, Claim(text=line, cell=cid, quantity=q, damage_class=cls)))
    return claims


def load_ev(tid, evidence_dir):
    return json.load(open(f"{evidence_dir}/{tid}.json"))


def claims_table(caps, evidence_dir, model_name):
    """One row per claim, with the checker's verdict and the evidence fraction for the claimed class."""
    rows = []
    for tid, cells in caps.items():
        ev = load_ev(tid, evidence_dir)
        by_cell = {c["cell"]: c for c in ev["cells"]}
        claims = [cl for _, cl in build_claims({tid: cells})]
        for c in verify_claims(claims, ev):
            cell = by_cell.get(c.cell)
            n = cell["buildings"] if cell else 0
            rows.append({"model": model_name, "tile_id": tid, "cell": c.cell,
                         "quantity": c.quantity, "damage_class": c.damage_class,
                         "verdict": c.verdict, "buildings": n,
                         "evidence_frac": round(class_fraction([cell], c.damage_class), 3) if n else None,
                         "text": c.text})
    return pd.DataFrame(rows, columns=TABLE_COLUMNS)


def summarize(table):
    """Verdict counts and rates for one claims table."""
    n = len(table)
    v = table["verdict"].value_counts()
    sup, con, nv = (int(v.get(k, 0)) for k in VERDICTS)
    checkable = sup + con
    return {"claims": n, "supported": sup, "contradicted": con, "not_verifiable": nv,
            "contradicted_rate": round(con / checkable, 3) if checkable else None,
            "not_verifiable_share": round(nv / n, 3) if n else None}


def presence_table(caps, evidence_dir):
    """Caption says 'no buildings' or not, against whether the evidence has buildings in that cell."""
    t = collections.Counter()
    for tid, cells in caps.items():
        ev = {c["cell"]: c for c in load_ev(tid, evidence_dir)["cells"]}
        for cid, line in cells.items():
            says_none = parse_line(line)[0] == "no_buildings"
            has = ev[cid]["buildings"] > 0
            t[("caption: no buildings" if says_none else "caption: buildings/other",
               "evidence: buildings" if has else "evidence: none")] += 1
    return t


def empty_baseline(caps, evidence_dir):
    """Presence accuracy of a captioner that answers 'no buildings' for every cell."""
    z = tot = 0
    for tid in caps:
        for c in load_ev(tid, evidence_dir)["cells"]:
            tot += 1
            z += c["buildings"] == 0
    return round(z / tot, 3)


def damage_detection(caps, evidence_dir):
    """Per cell with buildings: did the caption claim damage, and does the evidence have any?"""
    said, truth = {}, {}
    for tid, cl in build_claims(caps):
        cell = [c for c in load_ev(tid, evidence_dir)["cells"] if c["cell"] == cl.cell]
        if not cell or cell[0]["buildings"] == 0:
            continue
        key = (tid, cl.cell)
        said[key] = said.get(key, False) or cl.quantity != "none"
        truth[key] = class_fraction(cell, "damaged") > 0
    tp = sum(said[k] and truth[k] for k in said)
    fp = sum(said[k] and not truth[k] for k in said)
    fn = sum((not said[k]) and truth[k] for k in said)
    tn = sum((not said[k]) and not truth[k] for k in said)
    return {"TP": tp, "FP": fp, "FN": fn, "TN": tn,
            "precision": round(tp / (tp + fp), 3) if tp + fp else None,
            "recall": round(tp / (tp + fn), 3) if tp + fn else None}


def model_summary(name, caps, evidence_dir):
    p = presence_table(caps, evidence_dir)
    total = sum(p.values())
    agree = sum(k for (a, b), k in p.items() if (a == "caption: no buildings") == (b == "evidence: none"))
    d = damage_detection(caps, evidence_dir)
    row = {"model": name, "tiles": len(caps), "cells": total,
           "presence_agree": round(agree / total, 3),
           "always_empty_baseline": empty_baseline(caps, evidence_dir)}
    row.update(summarize(claims_table(caps, evidence_dir, name)))
    row.update({"damage_precision": d["precision"], "damage_recall": d["recall"]})
    return row

## Compare the two models on the same 20 tiles

The main run has 50 tiles; the fair comparison restricts it to the 20 tiles both models captioned. With 20 tiles the differences are indicative only, so say that in the report. `always_empty_baseline` is the presence accuracy of a captioner that writes "no buildings" for every cell; a model has to beat it to be of any use.

In [ ]:
caps_main_20 = {t: caps_main[t] for t in compare_ids}

tab_main = claims_table(caps_main, EVIDENCE_DIR, "Qwen3-VL-4B (50 tiles)")
tab_main_20 = claims_table(caps_main_20, EVIDENCE_DIR, "Qwen3-VL-4B (same 20)")
tab_cmp = claims_table(caps_cmp, EVIDENCE_DIR, "Qwen2.5-VL-3B (20 tiles)")

summary = pd.DataFrame([model_summary("Qwen3-VL-4B (50 tiles)", caps_main, EVIDENCE_DIR),
                        model_summary("Qwen3-VL-4B (same 20)", caps_main_20, EVIDENCE_DIR),
                        model_summary("Qwen2.5-VL-3B (20 tiles)", caps_cmp, EVIDENCE_DIR)]).set_index("model")
summary.T

## Plot (separate from the computation)

In [ ]:
import matplotlib.pyplot as plt


def verdict_shares(tables):
    """Share of claims per verdict, one list per table."""
    return [[((t["verdict"] == v).sum() / len(t)) if len(t) else 0.0 for v in VERDICTS] for t in tables]


def plot_verdict_shares(shares, names):
    """Stacked horizontal bars: share of supported / contradicted / not_verifiable per model."""
    fig, ax = plt.subplots(figsize=(7, 2 + 0.6 * len(names)))
    left = [0.0] * len(names)
    for j, v in enumerate(VERDICTS):
        vals = [s[j] for s in shares]
        ax.barh(names, vals, left=left, label=v)
        left = [a + b for a, b in zip(left, vals)]
    ax.set_xlim(0, 1)
    ax.set_xlabel("share of claims")
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.25), ncol=3)
    plt.tight_layout()
    plt.show()


plot_verdict_shares(verdict_shares([tab_main_20, tab_cmp]), ["Qwen3-VL-4B", "Qwen2.5-VL-3B"])

## Save the results

The tables feed the chain metrics and the human-check sheets (`flatten_claims` in notebook 04). `captioning.yaml` records the experiment settings; copy it into `configs/nlp/` in the repo.

In [ ]:
import yaml, datetime, transformers


def write_captioning_config(path):
    cfg = {
        "date": datetime.date.today().isoformat(),
        "transformers": str(transformers.__version__),
        "torch": str(torch.__version__),
        "main_model": MAIN_MODEL,
        "compare_model": COMPARE_MODEL,
        "dtype": "float16",
        "decoding": {"do_sample": False, "max_new_tokens": 40},
        "granularity": "per-cell crops (3x3), pre+post image pair per crop",
        "input_method": {"main_model": "images inside the chat message (caption_cell)",
                         "compare_model": "image placeholders + processor(images=...) (caption_cell_split)"},
        "seed": 42,
        "n_main": len(main_ids),
        "n_compare": len(compare_ids),
        "selection": "stratified by disaster, pilot tiles excluded; comparison = 4 tiles per disaster from the main set",
        "tiles_dir": TILES,
        "pilot_tiles": list(pilot_ids),
        "main_tiles": list(main_ids),
        "compare_tiles": list(compare_ids),
        "prompt": CELL_PROMPT,
    }
    with open(path, "w") as f:
        yaml.safe_dump(cfg, f, sort_keys=False, allow_unicode=True)
    return cfg


write_captioning_config(CFG_PATH)
pd.concat([tab_main, tab_cmp]).to_csv(f"{CAPTIONS}/claims_verdicts.csv", index=False)
summary.to_csv(f"{CAPTIONS}/model_comparison_summary.csv")
print("saved to", CAPTIONS)

## How to read this in the report

- `presence_agree` against `always_empty_baseline`: does the captioner find where the buildings are? Report both numbers.
- `contradicted_rate` is the check on the captioner: the share of checkable claims that the evidence contradicts. Report it with a bootstrap interval over tiles once you have more than a few hundred claims.
- `damage_precision` and `damage_recall` are per cell with buildings: did the caption claim any damage, and does the evidence have any.
- A high `not_verifiable_share` means the captions talk about cells the evidence cannot test (for example claims about cells with no buildings in the evidence).
- The evidence is derived from annotator labels, so a verdict means "consistent with the labelled evidence", not "visible in the image". Haze and cloud are not captured by `valid_fraction`.
- Both models are zero-shot and neither was trained on satellite damage. Say so, and do not describe either as accurate.

In [ ]:
model, processor = load_captioner(COMPARE_MODEL)
pre, post = crop_cells(Path(TILES) / "hurricane-matthew_00000152_p1")["r1c1"]
messages = [{"role": "user", "content": [{"type": "image"}, {"type": "image"},
            {"type": "text", "text": "Describe the buildings you can see in Image 2 and any damage."}]}]
text = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
inputs = processor(text=[text], images=[pre, post], return_tensors="pt").to(model.device)
tok = getattr(model.config, "image_token_id", None)
print("image_token_id:", tok, "| image tokens in input:", int((inputs["input_ids"] == tok).sum()) if tok else "n/a")
print(_generate(model, processor, inputs, 120))
del model, processor
free_gpu()

In [ ]:
cfg = yaml.safe_load(open(CFG_PATH))
cfg["compare_model_note"] = ("Qwen2.5-VL-3B received the images (72 image tokens per crop pair) and describes "
                             "buildings when asked a plain question, but with the constrained cell prompt it "
                             "returned 'no buildings.' for all 180 cells, so no claims could be verified.")
yaml.safe_dump(cfg, open(CFG_PATH, "w"), sort_keys=False, allow_unicode=True)